# XTTS v2 — 본인 음성 Fine-Tuning

**목표**: 본인 음성 30~45분(300건 내외)을 XTTS v2에 fine-tune해서 본인 전용 high-fidelity 모델 생성.

**KSS와 다른 점**:
- 데이터 적음 (300건 vs 5000건) → epoch 크게 잡음
- 단일 화자라 zero-shot 능력은 의도적으로 줄임 (본인 전용)
- 결과: 본인 목소리·말투·억양까지 학습한 모델

**전제**:
- 로컬에서 녹음 + `prepare_personal.py`로 manifest 생성 완료
- Drive `xtts_personal/{my_voice, manifest}` 업로드

In [ ]:
!nvidia-smi | head -10

In [ ]:
# Anti-idle
from IPython.display import display, Javascript
display(Javascript('setInterval(()=>{const b=document.querySelector("colab-toolbar-button#connect");if(b)b.click();},60000)'))
print('Anti-idle installed')

In [ ]:
# Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/xtts_personal'
DATASET    = f'{DRIVE_ROOT}/my_voice'
MANIFEST   = f'{DRIVE_ROOT}/manifest'
OUTPUT     = f'{DRIVE_ROOT}/runs'
os.makedirs(OUTPUT, exist_ok=True)

assert os.path.exists(f'{MANIFEST}/metadata_train.csv'), 'manifest 없음 — prepare_personal.py 먼저'
import glob
wavs = glob.glob(f'{DATASET}/*.wav')
print(f'WAV files: {len(wavs)} · 경로: {DATASET}')

In [ ]:
# Coqui TTS 설치
!pip install -q "git+https://github.com/idiap/coqui-ai-TTS.git"
!pip install -q pandas soundfile librosa
import TTS, torch
print('TTS:', TTS.__version__, '· CUDA:', torch.cuda.is_available())

In [ ]:
# Pretrained XTTS v2
from TTS.utils.manage import ModelManager
mm = ModelManager()
model_path, config_path, _ = mm.download_model('tts_models/multilingual/multi-dataset/xtts_v2')
PRETRAINED_DIR = os.path.dirname(model_path)
print('Pretrained:', PRETRAINED_DIR)

In [ ]:
# Config — 적은 데이터·많은 epoch
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.config.shared_configs import BaseDatasetConfig

config = XttsConfig()
config.load_json(os.path.join(PRETRAINED_DIR, 'config.json'))

# 본인 음성 fine-tune 설정
config.epochs           = 20            # 300건 × 20 epoch ≈ 6000 step
config.batch_size       = 2
config.eval_batch_size  = 1
config.grad_accum_steps = 4
config.mixed_precision  = True
config.lr               = 1e-5          # KSS보다 약간 높음 (적은 데이터)
config.optimizer        = 'AdamW'
config.optimizer_params = {'betas':[0.9,0.96], 'eps':1e-8, 'weight_decay':1e-2}
config.save_step        = 200
config.save_n_checkpoints = 2
config.print_step       = 25
config.run_name         = 'xtts_personal'
config.output_path      = OUTPUT

dataset = BaseDatasetConfig(
    formatter='ljspeech',
    meta_file_train=f'{MANIFEST}/metadata_train.csv',
    meta_file_val=f'{MANIFEST}/metadata_val.csv',
    path=DATASET, language='ko',
)
config.datasets = [dataset]
print(f'epochs={config.epochs} · batch={config.batch_size}*{config.grad_accum_steps}={config.batch_size*config.grad_accum_steps} · lr={config.lr}')

In [ ]:
# 모델 + Trainer (자동 resume)
from trainer import Trainer, TrainerArgs
from TTS.tts.models.xtts import Xtts

model = Xtts.init_from_config(config)
model.load_checkpoint(config, checkpoint_dir=PRETRAINED_DIR)

restore = None
ckpts = sorted(glob.glob(f'{OUTPUT}/*/checkpoint_*.pth'))
if ckpts:
    restore = ckpts[-1]
    print(f'[RESUME] {restore}')
else:
    print('[FRESH] start from pretrained')

trainer = Trainer(
    TrainerArgs(restore_path=restore),
    config, output_path=OUTPUT, model=model,
)

In [ ]:
# 학습 실행
# 300건 × 20 epoch ≈ 2~4시간 (Free T4)
trainer.fit()

In [ ]:
# Best 체크포인트 확인
best = sorted(glob.glob(f'{OUTPUT}/*/best_model.pth'))
print('Best:', best[-1] if best else 'not found')
print('All :', sorted(glob.glob(f'{OUTPUT}/*/*.pth'))[-3:])

In [ ]:
# 빠른 청취 테스트 — Base vs Fine-tuned
import IPython.display as ipd
from TTS.api import TTS

TEST_TEXT = '안녕하세요, 오늘은 정말 좋은 날입니다.'
REF_WAV = sorted(glob.glob(f'{DATASET}/*.wav'))[0]  # 본인 음성 첫 파일

# Base
tts_base = TTS('tts_models/multilingual/multi-dataset/xtts_v2', gpu=True)
tts_base.tts_to_file(text=TEST_TEXT, file_path='/content/base.wav',
                     speaker_wav=REF_WAV, language='ko')
print('Base (zero-shot, 6초 참조):')
ipd.display(ipd.Audio('/content/base.wav'))

# Fine-tuned
if best:
    from TTS.tts.models.xtts import Xtts
    cfg = XttsConfig(); cfg.load_json(os.path.join(os.path.dirname(best[-1]), 'config.json'))
    m = Xtts.init_from_config(cfg)
    m.load_checkpoint(cfg, checkpoint_path=best[-1])
    m.cuda().eval()
    out = m.synthesize(TEST_TEXT, cfg, speaker_wav=REF_WAV, language='ko', gpt_cond_len=3, temperature=0.7)
    import soundfile as sf
    sf.write('/content/tuned.wav', out['wav'], 24000)
    print('Fine-tuned (본인 음성 학습):')
    ipd.display(ipd.Audio('/content/tuned.wav'))